# YOLOV10 — Benchmark on OOD Dataset (22 classes)


In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [4]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov10n", "yolov10s", "yolov10m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov10.csv"
PERCLASS_CSV = "benchmark_yolov10_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov10n.pt",
    "yolov10s.pt",
    "yolov10m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


In [5]:
import cv2
try:
    from sahi.predict import get_sliced_prediction
    from sahi import AutoDetectionModel
except:
    pass

def preprocess_image(img_path):
    img = cv2.imread(str(img_path))
    if img is None: return None
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    final = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)
    return final

def benchmark_model(model_name):
    print(f'\n{"="*60}\n  BENCHMARK: {model_name}\n{"="*60}')
    model = YOLO(model_name)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=DEVICE, workers=WORKERS, name=f"{model_name.replace('.pt','')}_ood", patience=PATIENCE, save=True)
    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))
    metrics = best_model.val(data=DATA_YAML, split='test', device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS)
    
    # Enhanced inference loop
    test_img_dir = TEST_IMG_DIR
    test_images = [p for p in test_img_dir.glob('*.*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}][:200]
    sahi_model = None
    if USE_SAHI:
        device_str = f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu'
        sahi_model = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(best_path), device=device_str, confidence_threshold=0.25)
    
    tn, fp, neg_total, lats = 0, 0, 0, []
    for img_p in test_images:
        inp = preprocess_image(img_p) if USE_PREPROCESSING else str(img_p)
        t0 = time.perf_counter()
        if USE_SAHI and sahi_model:
            res = get_sliced_prediction(inp, sahi_model, slice_height=SAHI_SLICE_SIZE, slice_width=SAHI_SLICE_SIZE, overlap_height_ratio=SAHI_OVERLAP, overlap_width_ratio=SAHI_OVERLAP, verbose=0)
            cnt = len(res.object_prediction_list)
        else:
            res = best_model(inp, verbose=False)
            cnt = len(res[0].boxes) if res[0].boxes else 0
        lats.append((time.perf_counter()-t0)*1000)
        lbl_p = Path(str(img_p).replace('images', 'labels')).with_suffix('.txt')
        if not lbl_p.exists() or lbl_p.stat().st_size == 0:
            neg_total += 1
            if cnt == 0: tn += 1
            else: fp += 1
    
    row = {
        'model': model_name.replace('.pt',''),
        'mAP@0.5': round(float(metrics.box.map50), 4),
        'speed_ms/img': round(float(np.mean(lats)), 2),
        'neg_acc': round(tn/neg_total if neg_total > 0 else 1.0, 4),
        'size_MB': round(best_path.stat().st_size/1e6, 1),
    }
    del model, best_model
    gc.collect(); _safe_cuda_empty_cache()
    return row, {}


In [6]:
rows = []
all_per_class = {}

if RUN_TRAINING:
    print("RUN_TRAINING=True — training each model in MODELS (slow).\n")
    for model_name in MODELS:
        try:
            row, per_class = benchmark_model(model_name)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
            print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
            print(f"  Precision     : {row['precision']}")
            print(f"  Recall        : {row['recall']}")
            print(f"  F1            : {row['F1']}")
            print(f"  Speed         : {row['speed_ms/img']} ms/img")
            print(f"  Size          : {row['size_MB']} MB")
            print(f"  Params        : {row['params_M']} M")
        except Exception as e:
            print(f"  SKIPPED {model_name}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()
else:
    print("RUN_TRAINING=False — loading each variant's newest best.pt (no training).\n")
    print(f"Resolved runs/detect -> {_runs_detect_dir()}\n")
    for variant in BENCHMARK_VARIANTS:
        wp = _find_newest_best_pt(variant)
        if wp is None:
            print(f"  Skip {variant}: no matching best.pt under {_runs_detect_dir()} (expect folder '{variant}' or '{variant}_*')")
            continue
        try:
            row, per_class = metrics_from_weights(wp, variant)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  {variant}: mAP@0.5={row['mAP@0.5']}  speed={row['speed_ms/img']} ms/img")
        except Exception as e:
            print(f"  SKIPPED {variant}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()

_cols = [
    "model", "mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1",
    "speed_ms/img", "size_MB", "params_M",
]
df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)
df.to_csv(RESULTS_CSV, index=False)

if all_per_class:
    df_pc = pd.DataFrame(all_per_class).T
else:
    df_pc = pd.DataFrame(columns=CLASS_NAMES)
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV} ({len(df)} row(s))")
print(f"Saved -> {PERCLASS_CSV}")

RUN_TRAINING=False — loading each variant's newest best.pt (no training).

Resolved runs/detect -> C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect


====
  METRICS ONLY: yolov10n
  weights: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolov10n_ood\weights\best.pt
====
Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLOv10n summary (fused): 102 layers, 2,269,458 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 304.7203.2 MB/s, size: 395.8 KB)
val: Scanning C:\Users\admin\PFE\PFE-Obstacle-Detection\dataset\test\labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 8.4it/s 7.5s0.1s
                   all       1000       2945       0.68      0.543        0.6      0.422
                 bench         67        115      0.682      0.504  

In [7]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df.to_string(index=False))
styled = df.style.highlight_max(subset=['mAP@0.5', 'neg_acc'], color='#2d6a2e')
display(styled)


====
  YOLOv10 BENCHMARK — OOD Dataset (22 classes)
====

    model  mAP@0.5  mAP@0.5:0.95  precision  recall     F1  speed_ms/img  size_MB  params_M
yolov10n   0.5998        0.4224     0.6797  0.5426 0.6035         14.46      5.8       2.3
yolov10s   0.6099        0.4363     0.6829  0.5653 0.6186         17.08     49.0       7.2 



model,mAP@0.5,mAP@0.5:0.95,precision,recall,F1,speed_ms/img,size_MB,params_M
yolov10n,0.5998,0.4224,0.6797,0.5426,0.6035,14.5 ms,5.8 MB,2.3 M
yolov10s,0.6099,0.4363,0.6829,0.5653,0.6186,17.1 ms,49.0 MB,7.2 M


In [8]:
df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)

print("Per-class mAP@0.5 across YOLOv10 variants")
print("-" * 60)

styled_pc = (
    df_pc.style
    .set_caption("Per-Class mAP@0.5 — YOLOv10 Benchmark")
    .format("{:.4f}")
    .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
    .set_properties(**{"text-align": "center", "font-size": "12px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
    ])
)
display(styled_pc)

Per-class mAP@0.5 across YOLOv10 variants
------------------------------------------------------------


,bench,bicycle,bus,bus_stop,car,crutch,curb,dog,fire_hydrant,motorcycle,person,pole,spherical_roadblock,stairs,stop_sign,street_light,traffic_light,train,tree,truck,warning_column,waste_container
model,,,,,,,,,,,,,,,,,,,,,,
yolov10n,0.5488,0.5174,0.7073,0.9307,0.6104,0.4733,0.3831,0.6944,0.8211,0.6371,0.2122,0.3297,0.9143,0.6255,0.9087,0.2345,0.4828,0.7675,0.1620,0.6284,0.7948,0.8116
yolov10s,0.5374,0.5309,0.7456,0.9641,0.6340,0.5869,0.4340,0.6374,0.8336,0.6127,0.1854,0.3049,0.9147,0.6009,0.9107,0.2961,0.4047,0.8008,0.1407,0.6816,0.8285,0.8309


In [9]:
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(df["model"], df["mAP@0.5"], color="#2d6a2e")
axes[0].set_xlabel("mAP@0.5")
axes[0].set_title("Accuracy (mAP@0.5)")
axes[0].set_xlim(0, 1)

axes[1].barh(df["model"], df["speed_ms/img"], color="#1a5276")
axes[1].set_xlabel("ms / image")
axes[1].set_title("Inference Speed")

axes[2].barh(df["model"], df["size_MB"], color="#c0392b")
axes[2].set_xlabel("MB")
axes[2].set_title("Model Size")

plt.suptitle("YOLOv10 Benchmark — Indoor Dataset", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("benchmark_yolov10_chart.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1800x500 with 3 Axes>

In [10]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov10n", "yolov10s", "yolov10m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov10.csv"
PERCLASS_CSV = "benchmark_yolov10_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov10n.pt",
    "yolov10s.pt",
    "yolov10m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


Best model: yolov10s — loading ../runs/detect/yolov10s_ood/weights/best.pt


<Figure size 1800x1800 with 9 Axes>